In [2]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [3]:
from pathlib import Path
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np
import pickle
import math
import cv2


from notebooks.imports import *
from src.utils import pmf_utils

### Config

In [4]:
from config import dir_config

raw_dir = Path(dir_config.data.raw)
processed_dir = Path(dir_config.data.processed)

processed_dir = Path("/mnt/data/processed")

In [5]:
data = pd.read_csv(Path(processed_dir, "valid_data.csv"))

In [5]:
import os
import pickle
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

motion_data_dir = Path(processed_dir, "motion_data_dict")
progress_interval = 100  # Print every N files


def load_pickle(file_path):
	try:
		if os.path.getsize(file_path) == 0:
			return None
		with open(file_path, "rb") as f:
			trial_dict = pickle.load(f)
		trial_name = Path(file_path).stem.rsplit("me_array_", 1)[-1]
		return trial_name, trial_dict
	except Exception:
		return None


# Gather all .pkl files
all_files = list(motion_data_dir.glob("*.pkl"))
motion_data_dict = {}

# Thread-based execution for better I/O performance
completed = 0
print(f"🚀 Starting to load {len(all_files)} files...")

with ThreadPoolExecutor() as executor:
	futures = {executor.submit(load_pickle, file): file for file in all_files}
	for future in as_completed(futures):
		result = future.result()
		if result:
			trial_name, trial_dict = result
			motion_data_dict[trial_name] = trial_dict

		completed += 1
		if completed % progress_interval == 0:
			print(f"✅ Loaded {completed}/{len(all_files)} files...")

print(f"🎉 Done! Successfully loaded {len(motion_data_dict)} trial dicts.")


# save the motion data dict
with open(Path(processed_dir, "all_motion_energies.pkl"), "wb") as f:
	pickle.dump(motion_data_dict, f)


print("File saved successfully.")

🚀 Starting to load 113874 files...


✅ Loaded 100/113874 files...
✅ Loaded 200/113874 files...
✅ Loaded 300/113874 files...
✅ Loaded 400/113874 files...
✅ Loaded 500/113874 files...
✅ Loaded 600/113874 files...
✅ Loaded 700/113874 files...
✅ Loaded 800/113874 files...
✅ Loaded 900/113874 files...
✅ Loaded 1000/113874 files...
✅ Loaded 1100/113874 files...
✅ Loaded 1200/113874 files...
✅ Loaded 1300/113874 files...
✅ Loaded 1400/113874 files...
✅ Loaded 1500/113874 files...
✅ Loaded 1600/113874 files...
✅ Loaded 1700/113874 files...
✅ Loaded 1800/113874 files...
✅ Loaded 1900/113874 files...
✅ Loaded 2000/113874 files...
✅ Loaded 2100/113874 files...
✅ Loaded 2200/113874 files...
✅ Loaded 2300/113874 files...
✅ Loaded 2400/113874 files...
✅ Loaded 2500/113874 files...
✅ Loaded 2600/113874 files...
✅ Loaded 2700/113874 files...
✅ Loaded 2800/113874 files...
✅ Loaded 2900/113874 files...
✅ Loaded 3000/113874 files...
✅ Loaded 3100/113874 files...
✅ Loaded 3200/113874 files...
✅ Loaded 3300/113874 files...
✅ Loaded 3400/11387

In [6]:
with open(Path(processed_dir, "all_motion_energies.pkl"), "rb") as f:
	motion_data_dict = pickle.load(f)

In [7]:
def pad_motion_energy(me, pad_length=605):
	"""Pads motion energy to a fixed length, filling with NaNs."""
	if hasattr(me, "cpu"):
		me = me.cpu().numpy()
	me = np.asarray(me).flatten()  # Flatten handles both 1D and 2D inputs
	pad_me = np.full(pad_length, np.nan)
	pad_me[: len(me)] = me
	return pad_me


def normalize_data(session_data):
	"""Normalize each motion energy session data."""
	session_data = np.vstack(session_data)
	session_data /= np.nanmax(np.abs(session_data))  # Normalize across all trials
	return session_data


# Process only the first session
for idx in data.idx_session.unique():
	temp_data = data[data.idx_session == idx]
	session_codes = temp_data.agg_code.values

	# Define motion energy types
	motion_energy_types = {"global": [], "center": [], "Q1": [], "Q2": [], "Q3": [], "Q4": []}

	# Collect all motion energy vectors
	for code in session_codes:
		motion_energy_types["global"].append(pad_motion_energy(motion_data_dict[code]["global"]))
		motion_energy_types["center"].append(pad_motion_energy(motion_data_dict[code]["center"]))

		for quadrant in ["Q1", "Q2", "Q3", "Q4"]:
			motion_energy_types[quadrant].append(pad_motion_energy(motion_data_dict[code]["quadrants"][quadrant]))

	# Normalize each set of motion energies
	for key in motion_energy_types:
		motion_energy_types[key] = normalize_data(motion_energy_types[key])

	# Update dictionary with normalized data
	for i, code in enumerate(session_codes):
		motion_data_dict[code]["global"] = motion_energy_types["global"][i]
		motion_data_dict[code]["center"] = motion_energy_types["center"][i]
		for quadrant in ["Q1", "Q2", "Q3", "Q4"]:
			motion_data_dict[code]["quadrants"][quadrant] = motion_energy_types[quadrant][i]

In [8]:
with open(Path(processed_dir, "normalized_motion_energies.pkl"), "wb") as f:
	pickle.dump(motion_data_dict, f)